# Kaggle March Machine Learning Mania Data Exploration

This notebook explores the Kaggle NCAA tournament data for both Men's and Women's basketball.

## Data Categories
- **Reference Tables** - Teams, cities, conferences
- **Season Config** - Tournament structure by year
- **Tournament Seeds & Slots** - Bracket information
- **Regular Season Results** - Compact and detailed game stats
- **Tournament Results** - NCAA tournament games
- **Rankings** - Massey Ordinals (multiple ranking systems)
- **Coaches** - Coaching history

In [1]:
import pandas as pd
from pathlib import Path

# Base path to Kaggle data
DATA_DIR = Path("../data/kaggle")

# List all available files
files = sorted(DATA_DIR.glob("*.csv"))
print(f"Available files ({len(files)} total):")
for f in files:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:45} {size_kb:>10.1f} KB")

Available files (38 total):
  Cities.csv                                           9.4 KB
  Conferences.csv                                      1.6 KB
  MConferenceTourneyGames.csv                        174.8 KB
  MGameCities.csv                                   2729.2 KB
  MMasseyOrdinals.csv                             119255.3 KB
  MMasseyOrdinals_2025_133_37S.csv                   290.3 KB
  MMasseyOrdinals_2025_133_56S.csv                   435.7 KB
  MNCAATourneyCompactResults.csv                      73.9 KB
  MNCAATourneyDetailedResults.csv                    134.9 KB
  MNCAATourneySeedRoundSlots.csv                      15.5 KB
  MNCAATourneySeeds.csv                               38.6 KB
  MNCAATourneySlots.csv                               50.5 KB
  MRegularSeasonCompactResults.csv                  5522.7 KB
  MRegularSeasonDetailedResults.csv                11550.7 KB
  MSeasons.csv                                         1.8 KB
  MSecondaryTourneyCompactResults.csv     

## Reference Tables

In [2]:
# Load reference tables
cities = pd.read_csv(DATA_DIR / "Cities.csv")
conferences = pd.read_csv(DATA_DIR / "Conferences.csv")
m_teams = pd.read_csv(DATA_DIR / "MTeams.csv")
w_teams = pd.read_csv(DATA_DIR / "WTeams.csv")

print(f"Cities: {len(cities)} locations")
print(f"Conferences: {len(conferences)} conferences")
print(f"Men's Teams: {len(m_teams)} teams")
print(f"Women's Teams: {len(w_teams)} teams")

Cities: 503 locations
Conferences: 51 conferences
Men's Teams: 380 teams
Women's Teams: 378 teams


In [3]:
# View teams
print("Men's Teams columns:", list(m_teams.columns))
m_teams.head(10)

Men's Teams columns: ['TeamID', 'TeamName', 'FirstD1Season', 'LastD1Season']


,TeamID,TeamName,FirstD1Season,LastD1Season
0,1101,Abilene Chr,2014,2025
1,1102,Air Force,1985,2025
2,1103,Akron,1985,2025
3,1104,Alabama,1985,2025
4,1105,Alabama A&M,2000,2025
5,1106,Alabama St,1985,2025
6,1107,SUNY Albany,2000,2025
7,1108,Alcorn St,1985,2025
8,1109,Alliant Intl,1985,1991
9,1110,American Univ,1985,2025


In [4]:
# Conferences
conferences.head(20)

,ConfAbbrev,Description
0,a_sun,Atlantic Sun Conference
1,a_ten,Atlantic 10 Conference
2,aac,American Athletic Conference
3,acc,Atlantic Coast Conference
4,aec,America East Conference
5,asc,American South Conference
6,awc,American West Conference
7,big_east,Big East Conference
8,big_eight,Big Eight Conference
9,big_sky,Big Sky Conference


In [5]:
# Team conference assignments
m_team_confs = pd.read_csv(DATA_DIR / "MTeamConferences.csv")
print(f"Men's team-conference records: {len(m_team_confs)}")

# Current season conferences
current_season = m_team_confs["Season"].max()
current_confs = m_team_confs[m_team_confs["Season"] == current_season]
print(f"\nTeams in {current_season} season: {len(current_confs)}")

# Teams per conference (current season)
conf_counts = current_confs.merge(conferences, on="ConfAbbrev").groupby("Description").size().sort_values(ascending=False)
conf_counts.head(15)

Men's team-conference records: 13388

Teams in 2025 season: 364


Description
Atlantic Coast Conference             18
Big Ten Conference                    18
Big 12 Conference                     16
Southeastern Conference               16
Atlantic 10 Conference                15
Sun Belt Conference                   14
Coastal Athletic Association          14
American Athletic Conference          13
Metro Atlantic Athletic Conference    13
Mid-American Conference               12
Southland Conference                  12
Southwest Athletic Conference         12
Missouri Valley Conference            12
Atlantic Sun Conference               12
Horizon League                        11
dtype: int64

## Tournament Seeds

In [6]:
# Load tournament seeds
m_seeds = pd.read_csv(DATA_DIR / "MNCAATourneySeeds.csv")
w_seeds = pd.read_csv(DATA_DIR / "WNCAATourneySeeds.csv")

print(f"Men's seeds: {len(m_seeds)} records ({m_seeds['Season'].min()}-{m_seeds['Season'].max()})")
print(f"Women's seeds: {len(w_seeds)} records ({w_seeds['Season'].min()}-{w_seeds['Season'].max()})")

Men's seeds: 2626 records (1985-2025)
Women's seeds: 1744 records (1998-2025)


In [7]:
# Parse seed number from seed string (e.g., "W01" -> 1)
m_seeds["SeedNum"] = m_seeds["Seed"].str.extract(r"(\d+)").astype(int)

# Most frequent tournament teams (Men's)
tourney_appearances = m_seeds.merge(m_teams, on="TeamID").groupby("TeamName").size().sort_values(ascending=False)
print("Most NCAA Tournament Appearances (Men's):")
tourney_appearances.head(20)

Most NCAA Tournament Appearances (Men's):


TeamName
Kansas            39
Duke              38
Arizona           36
North Carolina    36
Kentucky          34
Michigan St       34
Texas             31
Purdue            31
UCLA              30
Syracuse          29
Oklahoma          28
Xavier            28
Indiana           27
Illinois          27
Louisville        27
Gonzaga           27
Wisconsin         26
Villanova         25
Florida           25
Connecticut       25
dtype: int64

In [8]:
# Most 1-seeds
one_seeds = m_seeds[m_seeds["SeedNum"] == 1].merge(m_teams, on="TeamID")
print("Most #1 Seeds (Men's):")
one_seeds.groupby("TeamName").size().sort_values(ascending=False).head(15)

Most #1 Seeds (Men's):


TeamName
Kansas            16
North Carolina    15
Duke              15
Kentucky          10
Arizona            7
Connecticut        6
Michigan St        5
Purdue             5
Oklahoma           5
Gonzaga            5
Villanova          4
Ohio St            4
Illinois           4
Virginia           4
Stanford           3
dtype: int64

## Regular Season Results

In [9]:
# Load regular season data
m_regular_compact = pd.read_csv(DATA_DIR / "MRegularSeasonCompactResults.csv")
m_regular_detailed = pd.read_csv(DATA_DIR / "MRegularSeasonDetailedResults.csv")

print(f"Men's Regular Season (Compact): {len(m_regular_compact):,} games")
print(f"Men's Regular Season (Detailed): {len(m_regular_detailed):,} games")
print(f"\nSeasons: {m_regular_compact['Season'].min()}-{m_regular_compact['Season'].max()}")

Men's Regular Season (Compact): 192,930 games
Men's Regular Season (Detailed): 118,882 games

Seasons: 1985-2025


In [10]:
# Compact results columns
print("Compact columns:", list(m_regular_compact.columns))
m_regular_compact.head()

Compact columns: ['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc', 'NumOT']


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT
0,1985,20,1228,81,1328,64,N,0
1,1985,25,1106,77,1354,70,H,0
2,1985,25,1112,63,1223,56,H,0
3,1985,25,1165,70,1432,54,H,0
4,1985,25,1192,86,1447,74,H,0


In [11]:
# Detailed results columns (has shooting stats)
print("Detailed columns:", list(m_regular_detailed.columns))
m_regular_detailed.head()

Detailed columns: ['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc', 'NumOT', 'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF', 'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF']


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
0,2003,10,1104,68,1328,62,N,0,27,58,...,10,16,22,10,22,8,18,9,2,20
1,2003,10,1272,70,1393,63,N,0,26,62,...,24,9,20,20,25,7,12,8,6,16
2,2003,11,1266,73,1437,61,N,0,24,58,...,26,14,23,31,22,9,12,2,5,23
3,2003,11,1296,56,1457,50,N,0,18,38,...,22,8,15,17,20,9,19,4,3,23
4,2003,11,1400,77,1208,71,N,0,30,61,...,16,17,27,21,15,12,10,7,1,14


In [12]:
# Games per season
games_per_season = m_regular_compact.groupby("Season").size()
print("Games per season:")
print(f"  Min: {games_per_season.min()} ({games_per_season.idxmin()})")
print(f"  Max: {games_per_season.max()} ({games_per_season.idxmax()})")
print(f"  Recent seasons:")
games_per_season.tail(10)

Games per season:
  Min: 3737 (1985)
  Max: 5641 (2025)
  Recent seasons:


Season
2016    5369
2017    5395
2018    5405
2019    5463
2020    5328
2021    3855
2022    5345
2023    5602
2024    5607
2025    5641
dtype: int64

In [13]:
# Score analysis
m_regular_compact["ScoreDiff"] = m_regular_compact["WScore"] - m_regular_compact["LScore"]
m_regular_compact["TotalScore"] = m_regular_compact["WScore"] + m_regular_compact["LScore"]

print("Score Statistics (All-Time):")
print(f"  Avg Winner Score: {m_regular_compact['WScore'].mean():.1f}")
print(f"  Avg Loser Score: {m_regular_compact['LScore'].mean():.1f}")
print(f"  Avg Margin: {m_regular_compact['ScoreDiff'].mean():.1f}")
print(f"  Avg Total: {m_regular_compact['TotalScore'].mean():.1f}")

Score Statistics (All-Time):
  Avg Winner Score: 76.9
  Avg Loser Score: 64.8
  Avg Margin: 12.1
  Avg Total: 141.6


In [14]:
# Highest scoring games
top_games = m_regular_compact.nlargest(10, "TotalScore").merge(
    m_teams.rename(columns={"TeamID": "WTeamID", "TeamName": "Winner"}), on="WTeamID"
).merge(
    m_teams.rename(columns={"TeamID": "LTeamID", "TeamName": "Loser"}), on="LTeamID"
)
top_games[["Season", "Winner", "WScore", "Loser", "LScore", "TotalScore"]]

,Season,Winner,WScore,Loser,LScore,TotalScore
0,1989,Loy Marymount,181,Alliant Intl,150,331
1,1991,Loy Marymount,186,Alliant Intl,140,326
2,1989,Loy Marymount,162,Alliant Intl,144,306
3,2022,Houston Chr,149,McNeese St,144,293
4,1990,Loy Marymount,152,Alliant Intl,137,289
5,1990,LSU,148,Loy Marymount,141,289
6,1991,Oklahoma,172,Loy Marymount,112,284
7,1989,Loy Marymount,147,Gonzaga,136,283
8,1985,UNLV,142,Utah St,140,282
9,2003,St Francis NY,142,LIU Brooklyn,140,282


In [15]:
# Biggest blowouts
blowouts = m_regular_compact.nlargest(10, "ScoreDiff").merge(
    m_teams.rename(columns={"TeamID": "WTeamID", "TeamName": "Winner"}), on="WTeamID"
).merge(
    m_teams.rename(columns={"TeamID": "LTeamID", "TeamName": "Loser"}), on="LTeamID"
)
blowouts[["Season", "Winner", "WScore", "Loser", "LScore", "ScoreDiff"]]

,Season,Winner,WScore,Loser,LScore,ScoreDiff
0,2020,Utah,143,MS Valley St,49,94
1,1996,Tulsa,141,Prairie View,50,91
2,1986,North Carolina,129,Manhattan,45,84
3,1992,Arkansas,128,Bethune-Cookman,46,82
4,1993,Oklahoma,146,Florida A&M,65,81
5,1997,Minnesota,114,Alabama St,34,80
6,2021,Arkansas,142,MS Valley St,62,80
7,1990,Duke,130,Harvard,54,76
8,1999,Maryland,132,North Texas,57,75
9,2000,LSU,112,Grambling,37,75


In [16]:
# Overtime games
ot_games = m_regular_compact[m_regular_compact["NumOT"] > 0]
print(f"Overtime games: {len(ot_games):,} ({100*len(ot_games)/len(m_regular_compact):.1f}%)")
print(f"\nOT distribution:")
ot_games["NumOT"].value_counts().sort_index()

Overtime games: 7,847 (4.1%)

OT distribution:


NumOT
1    6531
2    1068
3     200
4      42
5       5
6       1
Name: count, dtype: int64

## NCAA Tournament Results

In [17]:
# Load tournament results
m_tourney = pd.read_csv(DATA_DIR / "MNCAATourneyCompactResults.csv")
m_tourney_detailed = pd.read_csv(DATA_DIR / "MNCAATourneyDetailedResults.csv")

print(f"Men's Tournament Games: {len(m_tourney):,}")
print(f"Seasons: {m_tourney['Season'].min()}-{m_tourney['Season'].max()}")

Men's Tournament Games: 2,518
Seasons: 1985-2024


In [18]:
# Tournament wins by team
tourney_wins = m_tourney.merge(
    m_teams.rename(columns={"TeamID": "WTeamID", "TeamName": "Winner"}), on="WTeamID"
).groupby("Winner").size().sort_values(ascending=False)

print("Most NCAA Tournament Wins (All-Time):")
tourney_wins.head(25)

Most NCAA Tournament Wins (All-Time):


Winner
Duke              105
North Carolina     98
Kansas             94
Kentucky           84
Connecticut        67
Michigan St        63
Arizona            58
Syracuse           57
UCLA               55
Michigan           51
Louisville         49
Florida            48
Villanova          47
Gonzaga            46
Purdue             40
Arkansas           38
Indiana            37
Oklahoma           37
Wisconsin          36
Ohio St            35
Maryland           35
Texas              34
Illinois           33
Georgetown         33
Xavier             30
dtype: int64

In [19]:
# Championship games (last game of each season)
def get_championship_game(season_df):
    return season_df.loc[season_df["DayNum"].idxmax()]

championships = m_tourney.groupby("Season").apply(get_championship_game, include_groups=False).reset_index()
championships = championships.merge(
    m_teams.rename(columns={"TeamID": "WTeamID", "TeamName": "Champion"}), on="WTeamID"
).merge(
    m_teams.rename(columns={"TeamID": "LTeamID", "TeamName": "RunnerUp"}), on="LTeamID"
)

print("National Championship Results (Recent):")
championships[["Season", "Champion", "WScore", "RunnerUp", "LScore"]].tail(15)

National Championship Results (Recent):


,Season,Champion,WScore,RunnerUp,LScore
24,2009,North Carolina,89,Michigan St,72
25,2010,Duke,61,Butler,59
26,2011,Connecticut,53,Butler,41
27,2012,Kentucky,67,Kansas,59
28,2013,Louisville,82,Michigan,76
29,2014,Connecticut,60,Kentucky,54
30,2015,Duke,68,Wisconsin,63
31,2016,Villanova,77,North Carolina,74
32,2017,North Carolina,71,Gonzaga,65
33,2018,Villanova,79,Michigan,62


In [20]:
# Championship count
print("National Championships by Team:")
championships.groupby("Champion").size().sort_values(ascending=False).head(15)

National Championships by Team:


Champion
Connecticut       6
Duke              5
North Carolina    4
Villanova         3
Kansas            3
Kentucky          3
Louisville        2
Florida           2
Baylor            1
Arizona           1
Arkansas          1
Maryland          1
Indiana           1
Michigan St       1
Michigan          1
dtype: int64

## Seed Performance Analysis

In [21]:
# Add seeds to tournament games
m_seeds["SeedNum"] = m_seeds["Seed"].str.extract(r"(\d+)").astype(int)

tourney_with_seeds = m_tourney.merge(
    m_seeds[["Season", "TeamID", "SeedNum"]].rename(columns={"TeamID": "WTeamID", "SeedNum": "WSeed"}),
    on=["Season", "WTeamID"]
).merge(
    m_seeds[["Season", "TeamID", "SeedNum"]].rename(columns={"TeamID": "LTeamID", "SeedNum": "LSeed"}),
    on=["Season", "LTeamID"]
)

# Upset = lower seed (higher number) beats higher seed
tourney_with_seeds["Upset"] = tourney_with_seeds["WSeed"] > tourney_with_seeds["LSeed"]
print(f"Total tournament games with seeds: {len(tourney_with_seeds)}")
print(f"Upsets: {tourney_with_seeds['Upset'].sum()} ({100*tourney_with_seeds['Upset'].mean():.1f}%)")

Total tournament games with seeds: 2518
Upsets: 695 (27.6%)


In [22]:
# Win rate by seed
seed_wins = tourney_with_seeds.groupby("WSeed").size()
seed_losses = tourney_with_seeds.groupby("LSeed").size()

seed_record = pd.DataFrame({
    "Wins": seed_wins,
    "Losses": seed_losses
}).fillna(0).astype(int)
seed_record["WinPct"] = seed_record["Wins"] / (seed_record["Wins"] + seed_record["Losses"])
seed_record

,Wins,Losses,WinPct
1,515,131,0.797214
2,363,151,0.706226
3,287,152,0.653759
4,243,154,0.612091
5,180,156,0.535714
6,163,155,0.512579
7,139,155,0.472789
8,111,155,0.417293
9,96,156,0.380952
10,96,157,0.379447


In [23]:
# Classic matchups: 1 vs 16, 2 vs 15, etc.
first_round = tourney_with_seeds[
    ((tourney_with_seeds["WSeed"] + tourney_with_seeds["LSeed"]) == 17) |
    ((tourney_with_seeds["WSeed"] == 8) & (tourney_with_seeds["LSeed"] == 9)) |
    ((tourney_with_seeds["WSeed"] == 9) & (tourney_with_seeds["LSeed"] == 8))
]

matchup_results = []
for high_seed in range(1, 9):
    low_seed = 17 - high_seed if high_seed <= 8 else 9
    if high_seed == 8:
        low_seed = 9
    
    games = tourney_with_seeds[
        ((tourney_with_seeds["WSeed"] == high_seed) & (tourney_with_seeds["LSeed"] == low_seed)) |
        ((tourney_with_seeds["WSeed"] == low_seed) & (tourney_with_seeds["LSeed"] == high_seed))
    ]
    
    high_wins = len(games[games["WSeed"] == high_seed])
    low_wins = len(games[games["WSeed"] == low_seed])
    
    matchup_results.append({
        "Matchup": f"{high_seed} vs {low_seed}",
        "HighSeedWins": high_wins,
        "LowSeedWins": low_wins,
        "HighSeedWinPct": high_wins / (high_wins + low_wins) if (high_wins + low_wins) > 0 else 0
    })

pd.DataFrame(matchup_results)

,Matchup,HighSeedWins,LowSeedWins,HighSeedWinPct
0,1 vs 16,154,2,0.987179
1,2 vs 15,145,11,0.929487
2,3 vs 14,133,23,0.852564
3,4 vs 13,123,33,0.788462
4,5 vs 12,101,55,0.647436
5,6 vs 11,95,61,0.608974
6,7 vs 10,95,60,0.612903
7,8 vs 9,75,81,0.480769


## Massey Ordinals (Rankings)

In [24]:
# Note: The full MMasseyOrdinals.csv is very large (117 MB)
# Load a smaller 2025 sample first
try:
    ordinals_sample = pd.read_csv(DATA_DIR / "MMasseyOrdinals_2025_133_56S.csv")
    print(f"Loaded 2025 sample: {len(ordinals_sample):,} records")
    print(f"\nColumns: {list(ordinals_sample.columns)}")
    print(f"\nRanking systems: {ordinals_sample['SystemName'].nunique()}")
    ordinals_sample.head(10)
except FileNotFoundError:
    print("Sample file not found, loading full ordinals (may take a moment)...")
    ordinals_sample = pd.read_csv(DATA_DIR / "MMasseyOrdinals.csv")
    ordinals_sample = ordinals_sample[ordinals_sample["Season"] == 2024].head(1000)

Loaded 2025 sample: 19,702 records

Columns: ['Season', 'RankingDayNum', 'SystemName', 'TeamID', 'OrdinalRank']

Ranking systems: 56


In [25]:
# View ranking systems
print("Ranking Systems:")
ordinals_sample["SystemName"].unique()

Ranking Systems:


array(['7OT', 'AP', 'BAR', 'BBT', 'BIH', 'BMN', 'BNZ', 'BWE', 'COL',
       'DCI', 'DII', 'DOK', 'DOL', 'DP', 'DUN', 'EBP', 'EMK', 'ESR',
       'FAS', 'HAS', 'INC', 'JJK', 'JNG', 'KPI', 'KPK', 'LAB', 'LMC',
       'LOG', 'MAS', 'MB', 'MGS', 'MMG', 'MOR', 'NET', 'NOL', 'OMY',
       'PAC', 'PGH', 'PIR', 'POM', 'REW', 'RMS', 'RPI', 'RT', 'RTH',
       'SMS', 'SPR', 'SRS', 'STY', 'TRK', 'TRP', 'USA', 'WAB', 'WIL',
       'WLK', 'WOL'], dtype=object)

In [26]:
# Top teams by a specific ranking system
system = ordinals_sample["SystemName"].unique()[0]
latest_day = ordinals_sample["RankingDayNum"].max()

top_ranked = ordinals_sample[
    (ordinals_sample["SystemName"] == system) & 
    (ordinals_sample["RankingDayNum"] == latest_day)
].merge(m_teams, on="TeamID").nsmallest(25, "OrdinalRank")

print(f"Top 25 by {system} (Day {latest_day}):")
top_ranked[["OrdinalRank", "TeamName"]]

Top 25 by 7OT (Day 133):


,OrdinalRank,TeamName
17,1,Auburn
74,2,Duke
3,3,Alabama
113,4,Houston
104,5,Gonzaga
89,6,Florida
274,7,St Mary's CA
148,8,Louisville
163,9,Memphis
283,10,Tennessee


## Coaching Data

In [27]:
# Load coaching data
coaches = pd.read_csv(DATA_DIR / "MTeamCoaches.csv")
print(f"Coaching records: {len(coaches):,}")
print(f"\nColumns: {list(coaches.columns)}")
coaches.head(10)

Coaching records: 13,533

Columns: ['Season', 'TeamID', 'FirstDayNum', 'LastDayNum', 'CoachName']


,Season,TeamID,FirstDayNum,LastDayNum,CoachName
0,1985,1102,0,154,reggie_minton
1,1985,1103,0,154,bob_huggins
2,1985,1104,0,154,wimp_sanderson
3,1985,1106,0,154,james_oliver
4,1985,1108,0,154,davey_whitney
5,1985,1109,0,154,freddie_goss
6,1985,1110,0,154,ed_tapscott
7,1985,1111,0,154,kevin_cantwell
8,1985,1112,0,154,lute_olson
9,1985,1113,0,154,bob_weinhauer


In [28]:
# Most seasons coached (unique team-seasons)
coach_seasons = coaches.groupby("CoachName")["Season"].nunique().sort_values(ascending=False)
print("Most Seasons Coached:")
coach_seasons.head(20)

Most Seasons Coached:


CoachName
jim_larranaga       39
jim_boeheim         39
rick_barnes         38
bob_huggins         38
mike_krzyzewski     38
leonard_hamilton    37
cliff_ellis         37
dana_altman         36
john_calipari       33
lon_kruger          33
bob_mckillop        33
fran_dunphy         33
roy_williams        33
bill_herrion        32
bill_self           32
kelvin_sampson      32
jeff_a_jones        32
rick_pitino         31
tubby_smith         31
herb_sendek         31
Name: Season, dtype: int64

In [29]:
# Coaches with most tournament wins
# Get coach for each tournament winning team
tourney_wins_by_coach = m_tourney.merge(
    coaches.rename(columns={"TeamID": "WTeamID"}),
    on=["Season", "WTeamID"]
)

# Filter to games where coach was active during tournament (DayNum typically > 130)
tourney_wins_by_coach = tourney_wins_by_coach[
    tourney_wins_by_coach["DayNum"] >= tourney_wins_by_coach["FirstDayNum"]
]

coach_tourney_wins = tourney_wins_by_coach.groupby("CoachName").size().sort_values(ascending=False)
print("Most NCAA Tournament Wins by Coach:")
coach_tourney_wins.head(20)

Most NCAA Tournament Wins by Coach:


CoachName
mike_krzyzewski    101
roy_williams        79
jim_boeheim         57
bill_self           57
john_calipari       57
tom_izzo            56
rick_pitino         54
jim_calhoun         48
lute_olson          44
mark_few            43
dean_smith          37
billy_donovan       35
jay_wright          34
bob_huggins         34
tubby_smith         30
rick_barnes         30
eddie_sutton        28
gary_williams       28
bo_ryan             27
steve_fisher        26
dtype: int64

## Detailed Statistics Analysis

In [30]:
# Analyze detailed stats from recent season
recent_season = m_regular_detailed["Season"].max()
recent_detailed = m_regular_detailed[m_regular_detailed["Season"] == recent_season].copy()

print(f"Analyzing {recent_season} season: {len(recent_detailed)} games with detailed stats")
print(f"\nStatistic columns:")
stat_cols = [c for c in recent_detailed.columns if c.startswith(("W", "L")) and c not in ["WTeamID", "LTeamID", "WScore", "LScore", "WLoc"]]
print(stat_cols)

Analyzing 2025 season: 5641 games with detailed stats

Statistic columns:
['WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF', 'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF']


In [31]:
# Calculate shooting percentages
recent_detailed["W_FG_Pct"] = recent_detailed["WFGM"] / recent_detailed["WFGA"]
recent_detailed["W_3P_Pct"] = recent_detailed["WFGM3"] / recent_detailed["WFGA3"]
recent_detailed["W_FT_Pct"] = recent_detailed["WFTM"] / recent_detailed["WFTA"]

print("Winner Shooting Stats:")
print(f"  FG%: {recent_detailed['W_FG_Pct'].mean():.1%}")
print(f"  3P%: {recent_detailed['W_3P_Pct'].mean():.1%}")
print(f"  FT%: {recent_detailed['W_FT_Pct'].mean():.1%}")

Winner Shooting Stats:
  FG%: 47.9%
  3P%: 37.0%
  FT%: 73.7%


In [32]:
# Team season stats
def calc_team_stats(team_id, detailed_df):
    """Calculate aggregate stats for a team."""
    as_winner = detailed_df[detailed_df["WTeamID"] == team_id]
    as_loser = detailed_df[detailed_df["LTeamID"] == team_id]
    
    wins = len(as_winner)
    losses = len(as_loser)
    
    # Points
    pts_for = as_winner["WScore"].sum() + as_loser["LScore"].sum()
    pts_against = as_winner["LScore"].sum() + as_loser["WScore"].sum()
    games = wins + losses
    
    return {
        "Wins": wins,
        "Losses": losses,
        "PPG": pts_for / games if games > 0 else 0,
        "OppPPG": pts_against / games if games > 0 else 0,
        "Margin": (pts_for - pts_against) / games if games > 0 else 0
    }

# Get stats for all teams in recent season
team_ids = set(recent_detailed["WTeamID"]).union(set(recent_detailed["LTeamID"]))
team_stats = []
for tid in team_ids:
    stats = calc_team_stats(tid, recent_detailed)
    stats["TeamID"] = tid
    team_stats.append(stats)

team_stats_df = pd.DataFrame(team_stats).merge(m_teams, on="TeamID")
team_stats_df["WinPct"] = team_stats_df["Wins"] / (team_stats_df["Wins"] + team_stats_df["Losses"])

# Top teams by point margin
print(f"Top Teams by Point Margin ({recent_season}):")
team_stats_df.nlargest(20, "Margin")[["TeamName", "Wins", "Losses", "WinPct", "PPG", "OppPPG", "Margin"]]

Top Teams by Point Margin (2025):


,TeamName,Wins,Losses,WinPct,PPG,OppPPG,Margin
74,Duke,31,3,0.911765,82.705882,61.911765,20.794118
104,Gonzaga,25,8,0.757576,86.636364,69.636364,17.000000
89,Florida,30,4,0.882353,85.411765,69.235294,16.176471
113,Houston,30,4,0.882353,74.205882,58.470588,15.735294
354,UC San Diego,28,4,0.875000,77.937500,62.843750,15.093750
159,Maryland,25,8,0.757576,81.666667,67.000000,14.666667
17,Auburn,28,5,0.848485,83.848485,69.606061,14.242424
318,VCU,27,6,0.818182,76.333333,62.515152,13.818182
289,Texas Tech,25,8,0.757576,80.909091,67.575758,13.333333
271,St John's,30,4,0.882353,78.705882,65.882353,12.823529


## Women's Basketball Quick Look

In [33]:
# Load women's data
w_regular = pd.read_csv(DATA_DIR / "WRegularSeasonCompactResults.csv")
w_tourney = pd.read_csv(DATA_DIR / "WNCAATourneyCompactResults.csv")

print(f"Women's Regular Season: {len(w_regular):,} games")
print(f"Women's Tournament: {len(w_tourney):,} games")
print(f"Seasons: {w_regular['Season'].min()}-{w_regular['Season'].max()}")

Women's Regular Season: 137,028 games
Women's Tournament: 1,650 games
Seasons: 1998-2025


In [34]:
# Women's tournament wins
w_tourney_wins = w_tourney.merge(
    w_teams.rename(columns={"TeamID": "WTeamID", "TeamName": "Winner"}), on="WTeamID"
).groupby("Winner").size().sort_values(ascending=False)

print("Most Women's NCAA Tournament Wins:")
w_tourney_wins.head(20)

Most Women's NCAA Tournament Wins:


Winner
Connecticut       116
Tennessee          76
Stanford           70
Notre Dame         68
Duke               57
Baylor             57
LSU                47
South Carolina     46
Louisville         42
Maryland           41
North Carolina     37
Purdue             36
Oklahoma           34
Georgia            31
Texas              30
Rutgers            28
Texas A&M          27
Iowa               24
Mississippi St     24
Florida St         23
dtype: int64

## Submission Format Preview

In [35]:
# Preview submission format
sample_sub = pd.read_csv(DATA_DIR / "SampleSubmissionStage2.csv")
print(f"Submission rows: {len(sample_sub):,}")
print(f"\nFormat: {sample_sub.columns.tolist()}")
print(f"\nID format examples:")
sample_sub.head(10)

Submission rows: 131,407

Format: ['ID', 'Pred']

ID format examples:


,ID,Pred
0,2025_1101_1102,0.5
1,2025_1101_1103,0.5
2,2025_1101_1104,0.5
3,2025_1101_1105,0.5
4,2025_1101_1106,0.5
5,2025_1101_1107,0.5
6,2025_1101_1108,0.5
7,2025_1101_1110,0.5
8,2025_1101_1111,0.5
9,2025_1101_1112,0.5


In [36]:
# Parse submission ID format
# Format: YYYY_TEAMID1_TEAMID2 (lower ID first)
sample_sub[["Season", "Team1", "Team2"]] = sample_sub["ID"].str.split("_", expand=True).astype(int)
print(f"Season: {sample_sub['Season'].unique()}")
print(f"Unique Team1 IDs: {sample_sub['Team1'].nunique()}")
print(f"Unique Team2 IDs: {sample_sub['Team2'].nunique()}")

Season: [2025]
Unique Team1 IDs: 724
Unique Team2 IDs: 724
